In [ ]:
# Import utility functions
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import requests
import json
sys.path.append('..')  # Add parent directory to path

# Import from local utility modules
from lvt_utils import (model_split_rate_tax, calculate_current_tax, ensure_geodataframe, 
                       categorize_property_type, calculate_category_tax_summary, print_category_tax_summary)
from census_utils import (get_census_data_with_boundaries, match_parcels_to_demographics, 
                          create_demographic_summary, calculate_median_percentage_by_quintile, match_to_census_blockgroups)
from viz import (create_scatter_plot, plot_comparison, calculate_correlations, weighted_median, 
                 create_quintile_summary, plot_quintile_analysis, create_property_category_chart, 
                 create_map_visualization, calculate_block_group_summary, filter_data_for_analysis)
from policy_analysis import analyze_land_by_improvement_share

# Control variable for data scraping
data_scrape = 0  # Set to 1 to scrape new data, 0 to use existing data

print("✅ Utility functions imported from LVTShift modules")

In [ ]:
import glob
from datetime import datetime

# Create data directory if it doesn't exist
data_dir = "data/denver/"
os.makedirs(data_dir, exist_ok=True)

In [ ]:
import geopandas as gpd
import pandas as pd

gdf_parcels = gpd.read_parquet("data/denver/1-assemble-universe.parquet")
#df_mills = pd.read_csv("data/denver/denver_millage_2024.csv")
df_mills = pd.read_csv("data/denver/denver_just_city_county_millage_2024.csv")

In [ ]:
# Enforce downtown district
gdf_district = gpd.read_parquet("data/denver/downtown_district.parquet")

In [ ]:
# 1. Make sure both GDFs are in the same CRS
gdf_district = gdf_district.to_crs(gdf_parcels.crs)

# 2. Build a centroid GDF (preserves original index)
gdf_centroids = gdf_parcels.set_geometry(gdf_parcels.geometry.centroid)

# 3. Spatial join: centroid-in-polygon
joined = gpd.sjoin(
    gdf_centroids[["geometry"]],          # only index — no extra columns needed
    gdf_district[["name", "geometry"]],  # only the field(s) you want to stamp
    how="left",
    predicate="within",
)

# 4. Handle duplicates (centroid on a shared boundary can match >1 district)
joined = joined[~joined.index.duplicated(keep="first")]

# 5. Stamp the result back onto the original parcel GDF
gdf_parcels["lvt_district"] = joined["name"]

In [ ]:
land_to_impr_ratio = 2

In [ ]:
df_mills

In [ ]:
gdf = gdf_parcels.merge(df_mills, on="tax_district", how="left")

In [ ]:
# Assessment formula is:

# (Actual Value * Assessed Rate) - Exemption = Taxable Value
# (Taxable Value) * Millage = Taxes Owed

# 2025 Assessment Rates

r_rate_local   = 0.0625 # residential-improved, non-school local government
r_rate_schools = 0.0705 # residential-improved, schools only
o_rate         = 0.27   # all other, all local governments

In [ ]:
for f in ["exempt_school_value", "exempt_local_value", "assr_market_value", "assr_land_value", "assr_impr_value"]:
    gdf[f] = gdf[f].fillna(0.0)

In [ ]:
# Guarantee land+impr values always add up to market value:

gdf["assr_impr_value"] = gdf["assr_market_value"] - gdf["assr_land_value"]

In [ ]:
gdf["is_tax_class_res"] = gdf["model_group"].eq("single_family") & gdf["is_vacant"].eq(False)

In [ ]:
gdf["is_vacant"] = (
    (~gdf["bldg_area_finished_sqft"].gt(0) & ~gdf["assr_impr_value"].gt(0)) |
    (gdf["model_group"].eq("single_family") & (gdf["assr_impr_value"]/gdf["assr_market_value"]).lt(0.1))
)

In [ ]:
gdf["PROPERTY_CATEGORY"] = np.where(gdf["is_vacant"], "vacant_land", gdf["model_group"])

In [ ]:
# Calculate taxable values for three cases:
# 1. Residential-improved for schools
# 2. Residential-improved for all other local government
# 3. All other categories for all local governments

# Calculate assessed values and subtract exemptions
gdf["taxable_value_r_schools"] = ((gdf["assr_market_value"]) * r_rate_schools)
gdf["taxable_value_r_local"] = ((gdf["assr_market_value"]) * r_rate_local)
gdf["taxable_value_other"] = ((gdf["assr_market_value"]) * o_rate)

# Calculate millage rates
gdf["taxes_r_schools"] = gdf["taxable_value_r_schools"] * gdf["mills_school"] * 0.001
gdf["taxes_r_local"] = gdf["taxable_value_r_local"] * gdf["mills_local"] * 0.001
gdf["taxes_other"] = gdf["taxable_value_other"] * gdf["mills_all"] * 0.001

# Assign property tax based on class
gdf["property_tax"] = 0.0
gdf.loc[gdf["is_tax_class_res"].eq(True), "property_tax"] = gdf["taxes_r_schools"].astype(float) + gdf["taxes_r_local"].astype(float)
gdf.loc[gdf["is_tax_class_res"].eq(False), "property_tax"] = gdf["taxes_other"].astype(float)

gdf["property_tax"].sum()

In [ ]:
for word in ["land", "impr", "market"]:
    gdf[f"taxable_value_{word}_school"] = np.where(
        gdf["is_tax_class_res"], 
        gdf[f"assr_{word}_value"] * r_rate_schools, 
        gdf[f"assr_{word}_value"] * o_rate
    )
    gdf[f"taxable_value_{word}_nonschool"] = np.where(
        gdf["is_tax_class_res"], 
        gdf[f"assr_{word}_value"] * r_rate_local, 
        gdf[f"assr_{word}_value"] * o_rate
    )

In [ ]:
# Exempt amount seems to be calculated from
# some unstated amount of the base value, times the assessment rate of local/school, respectively
# This means we can back out the exempted amount of actual value by dividing

# Divide the exemption amount by the appropriate tax rate to get the amount of actual value exempted
gdf["exempt_local_total"] = gdf["exempt_local_value"] / r_rate_local
gdf["exempt_school_total"] = gdf["exempt_school_value"] / r_rate_schools

# Get the difference between exempted value and actual value per class
gdf["exempt_diff_local"] = gdf["assr_market_value"] - gdf["exempt_local_total"]
gdf["exempt_diff_school"] = gdf["assr_market_value"] - gdf["exempt_school_total"]

# It seems fine to just use the local rate, because both _local and _school settle on similar figures when we reverse the operation

# Get the difference as percentage and absolute percentage of actual value
gdf["ex_diff_perc"] = (gdf["exempt_diff_local"]/gdf["assr_market_value"])
gdf["ex_diff_abs_perc"] = (gdf["exempt_diff_local"]/gdf["assr_market_value"]).abs()

# Assign amount exempted
gdf["exempt_perc"] = 0.0

# If the exempted value is higher than actual value, count that as fully exempt
gdf.loc[gdf["ex_diff_perc"].le(0), "exempt_perc"] = 1.0

# If the exempted value is lower than actual value, count it as partially exempt
gdf.loc[
    gdf["exempt_perc"].lt(1.0) &
    gdf["ex_diff_perc"].gt(0), "exempt_perc"
] = gdf["exempt_local_total"] / gdf["assr_market_value"]

# Anything with 100% exempt is flagged with a binary
gdf["is_exempt"] = False
gdf.loc[gdf["exempt_perc"].ge(1.0), "is_exempt"] = True
gdf.loc[gdf["assr_land_value"].eq(0.0) & gdf["assr_impr_value"].eq(0.0), "is_exempt"] = True

gdf["exemptions"] = gdf["assr_market_value"] * gdf["exempt_perc"]

#gdf.loc[gdf["PROPERTY_CATEGORY"].eq("single_family"), "exemptions"] = gdf["exemptions"] + 100000
#gdf.loc[gdf["PROPERTY_CATEGORY"].eq("vacant_land"), "exemptions"] = 0.0

# gdf["is_exempt"] = False
# gdf["exempt_perc"] = 0.0
# gdf["ex_diff_perc"] = 0.0
# gdf["exemptions"] = 0.0

from policy_analysis import analyze_property_values_by_category

analyze_property_values_by_category(gdf, "PROPERTY_CATEGORY", "taxable_value_land_nonschool", "taxable_value_impr_nonschool", "exemptions", "is_exempt")

In [ ]:
gdf[
    gdf["PROPERTY_CATEGORY"].eq("UNKNOWN") &
    gdf["is_exempt"]
][["address","assr_market_value","assr_land_value","assr_impr_value","bldg_area_finished_sqft","exemptions"]].sort_values(by="assr_market_value", ascending=False)

In [ ]:
gdf = gdf[gdf["is_exempt"].eq(False)].copy()

In [ ]:
# Exclude tiny categories
vcs = gdf["PROPERTY_CATEGORY"].value_counts()
gdf = gdf[
    gdf["PROPERTY_CATEGORY"].isin(vcs[vcs > 50].index.tolist()) &
    gdf["PROPERTY_CATEGORY"].ne("UNKNOWN")
].copy()

In [ ]:
# Calculate current tax:
field_s = f"taxable_value_market_school"
field_ns = f"taxable_value_market_nonschool"



In [ ]:
gdf.groupby("PROPERTY_CATEGORY")[["assr_land_value","assr_impr_value","exemptions"]].sum()

In [ ]:
# Calculate split-rate tax:
field_market_s = f"taxable_value_market_school"
field_land_s = f"taxable_value_land_school"
field_impr_s = f"taxable_value_impr_school"

field_market_ns = f"taxable_value_market_nonschool"
field_land_ns = f"taxable_value_land_nonschool"
field_impr_ns = f"taxable_value_impr_nonschool"


In [ ]:
gdfs = []

for tax_district in gdf["tax_district"].unique():
    gdf_td = gdf[gdf["tax_district"].eq(tax_district)].copy()

    # Calculate current tax for all parcels in the district
    current_tax_s, _, gdf_s = calculate_current_tax(gdf_td, field_s, "mills_school", "exemptions", "is_exempt", verbose=True)
    current_tax_ns, _, gdf_ns = calculate_current_tax(gdf_td, field_ns, "mills_local", "exemptions", "is_exempt", verbose=True)

    current_tax_total = current_tax_s + current_tax_ns
    gdf_td["current_tax_school"] = gdf_s["current_tax"]
    gdf_td["current_tax_nonschool"] = gdf_ns["current_tax"]
    gdf_td["current_tax"] = gdf_td["current_tax_school"] + gdf_td["current_tax_nonschool"]

    # Split: only LVT-district parcels get the split-rate model applied
    lvt_mask = gdf_td["lvt_district"].notna()
    gdf_lvt = gdf_td[lvt_mask].copy()
    gdf_rest = gdf_td[~lvt_mask].copy()

    if len(gdf_lvt) > 0:
        # Revenue targets are only the LVT parcels' share
        current_tax_s_lvt = gdf_lvt["current_tax_school"].sum()
        current_tax_ns_lvt = gdf_lvt["current_tax_nonschool"].sum()

        land_mills_s, impr_mills_s, new_tax_s, gdf_new_s = model_split_rate_tax(
            df=gdf_lvt,
            land_value_col=field_land_s,
            improvement_value_col=field_impr_s,
            current_revenue=current_tax_s_lvt,
            land_improvement_ratio=land_to_impr_ratio,
            exemption_col="exemptions",
            exemption_flag_col="is_exempt",
            verbose=True
        )

        land_mills_ns, impr_mills_ns, new_tax_ns, gdf_new_ns = model_split_rate_tax(
            df=gdf_lvt,
            land_value_col=field_land_ns,
            improvement_value_col=field_impr_ns,
            current_revenue=current_tax_ns_lvt,
            land_improvement_ratio=land_to_impr_ratio,
            exemption_col="exemptions",
            exemption_flag_col="is_exempt",
            verbose=True
        )

        gdf_lvt["new_tax_school"] = gdf_new_s["new_tax"]
        gdf_lvt["new_tax_nonschool"] = gdf_new_ns["new_tax"]
        gdf_lvt["new_tax"] = gdf_lvt["new_tax_school"] + gdf_lvt["new_tax_nonschool"]

        gdf_lvt["land_tax_school"] = gdf_new_s["land_tax"]
        gdf_lvt["land_tax_nonschool"] = gdf_new_ns["land_tax"]
        gdf_lvt["land_tax"] = gdf_lvt["land_tax_school"] + gdf_lvt["land_tax_nonschool"]

        gdf_lvt["improvement_tax_school"] = gdf_new_s["improvement_tax"]
        gdf_lvt["improvement_tax_nonschool"] = gdf_new_ns["improvement_tax"]
        gdf_lvt["improvement_tax"] = gdf_lvt["improvement_tax_school"] + gdf_lvt["improvement_tax_nonschool"]

    # Non-LVT parcels (and districts with no LVT parcels): new tax equals current tax
    for gdf_no_change in ([gdf_rest] + ([gdf_lvt] if len(gdf_lvt) == 0 else [])):
        gdf_no_change["new_tax_school"] = gdf_no_change["current_tax_school"]
        gdf_no_change["new_tax_nonschool"] = gdf_no_change["current_tax_nonschool"]
        gdf_no_change["new_tax"] = gdf_no_change["current_tax"]
        gdf_no_change["land_tax_school"] = gdf_no_change["current_tax_school"]
        gdf_no_change["land_tax_nonschool"] = gdf_no_change["current_tax_nonschool"]
        gdf_no_change["land_tax"] = gdf_no_change["current_tax"]
        gdf_no_change["improvement_tax_school"] = 0.0
        gdf_no_change["improvement_tax_nonschool"] = 0.0
        gdf_no_change["improvement_tax"] = 0.0

    gdfs.append(pd.concat([gdf_lvt, gdf_rest]))

gdf = pd.concat(gdfs)

print("")
print(f" SCHOOL CURR TAX  = {gdf['current_tax_school'].sum():>20,.2f}")
print(f"NSCHOOL CURR TAX  = {gdf['current_tax_nonschool'].sum():>20,.2f}")
print(f"  TOTAL CURR TAX  = {gdf['current_tax'].sum():>20,.2f}")
print("")
print(f" SCHOOL NEW   TAX = {gdf['new_tax_school'].sum():>20,.2f}")
print(f"NSCHOOL NEW   TAX = {gdf['new_tax_nonschool'].sum():>20,.2f}")
print(f"  TOTAL NEW   TAX = {gdf['new_tax'].sum():>20,.2f}")

display(gdf)

In [ ]:
# Calculate comprehensive tax impact summary by property category
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax_nonschool',
    new_tax_col='new_tax_nonschool',
    pct_threshold=10.0
)

# Print formatted summary
print_category_tax_summary(
    summary_df=category_summary,
    title=f"{land_to_impr_ratio:.0f}:1 Split-Rate Tax Impact by Property Category - Denver, CO",
    pct_threshold=10.0
)

# Display the detailed summary table
display(category_summary)


In [ ]:
# Calculate tax changes
gdf['tax_change'] = gdf['new_tax'] - gdf['current_tax']
gdf['tax_change_pct'] = (gdf['tax_change'] / gdf['property_tax']) * 100
gdf["tax_change_pct"] = gdf["tax_change_pct"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Add land price per square foot
gdf['land_price_per_sqft'] = gdf['assr_land_value'] / gdf['land_area_sqft']

In [ ]:
gdf[
    gdf["PROPERTY_CATEGORY"].eq("single_family")
][[
    "address",
    "assr_land_value",
    "assr_impr_value",
    "current_tax",
    "new_tax",
    "land_tax",
    "improvement_tax",
    "tax_change_pct",
    "mills_school",
    "mills_local",
    "mills_all",
]].sort_values(by="tax_change_pct", ascending=False)

In [ ]:
# Save the processed data with today's date in the filename
today_str = datetime.now().strftime("%Y-%m-%d")
save_path = f"{data_dir}denver_parcels_processed_{today_str}.parquet"
gdf.to_parquet(save_path)
print(f"💾 Saved processed data to {save_path}")

In [ ]:
print(f"\n📊 Dataset Overview:")
print(f"Total parcels: {len(gdf):,}")
print(f"Columns: {len(gdf.columns)}")
print(f"Geometry type: {gdf.geometry.geom_type.iloc[0]}")

# Display key statistics
total_current_revenue = gdf['current_tax'].sum()
print(f"\n💰 Current Tax System:")
print(f"Total annual revenue: ${total_current_revenue:,.2f}")
print(f"Mean property tax: ${gdf['current_tax'].mean():,.2f}")
print(f"Median property tax: ${gdf['current_tax'].median():.2f}")


In [ ]:
# Ensure proper GeoDataFrame format
gdf = ensure_geodataframe(gdf)

gdf["Lvalue"] = gdf["assr_land_value"]
gdf["Fvalue"] = gdf["assr_market_value"]

# Display property statistics
print(f"🏠 Property Statistics:")
print(f"Total parcels    : {len(gdf):10,}")
print(f"Vacant parcels   : {gdf['is_vacant'].sum():10,} ({gdf['is_vacant'].sum()/len(gdf)*100:4.1f}%)")

print("\n📋 Property Categories (Standardized):")
category_counts = gdf['PROPERTY_CATEGORY'].value_counts()
for category, count in category_counts.head(10).items():
    pct = count / len(gdf) * 100
    print(f"  {category:25}: {count:10,} ({pct:4.1f}%)")

print("\n🏗️ Property Value Statistics:")
print(f"Mean land        : ${gdf['Lvalue'].mean():20,.2f}")
print(f"Mean improvement : ${gdf['Fvalue'].mean():20,.2f}")
print(f"Total land       : ${gdf['Lvalue'].sum():20,.2f}")
print(f"Total improvement: ${gdf['Fvalue'].sum():20,.2f}")


In [ ]:
# Calculate current property taxes
total_revenue = gdf["current_tax"].sum()

print(f"Calculated total current tax revenue: ${total_revenue:,.2f}")

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
display(gdf.head())

In [ ]:
# Calculate comprehensive tax impact summary by property category
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    pct_threshold=10.0
)

# Print formatted summary
print_category_tax_summary(
    summary_df=category_summary,
    title=f"{land_to_impr_ratio:.0f}:1 Split-Rate Tax Impact by Property Category - Denver, CO",
    pct_threshold=10.0
)

# Display the detailed summary table
display(category_summary)


In [ ]:
# Property category impact chart (Spokane style, sorted, ignore 0% median)
import matplotlib.pyplot as plt
import numpy as np

# Filter out categories where median tax change percent is exactly 0
filtered = category_summary[category_summary['median_tax_change_pct'] != 0].copy()

# Only include categories with property_count > 0 (optional, but safe)
filtered = filtered[filtered['property_count'] > 0]

if filtered.empty:
    print('No categories with non-zero median tax change (LVT district may be empty or not set) — skipping chart.')
else:

    # Sort by median_pct_change ascending (like Spokane)
    categories = filtered['PROPERTY_CATEGORY'].tolist()
    counts = filtered['property_count'].tolist()
    median_pct_change = filtered['median_tax_change_pct'].tolist()
    median_dollar_change = filtered['median_tax_change'].tolist()
    total_tax_change = (
        filtered['total_tax_change_dollars'].tolist()
        if 'total_tax_change_dollars' in filtered.columns
        else (filtered['mean_tax_change'] * filtered['property_count']).tolist()
    )

    # Sort by median_pct_change ascending
    sorted_idx = np.argsort(median_pct_change)
    categories = [categories[i] for i in sorted_idx]
    counts = [counts[i] for i in sorted_idx]
    median_pct_change = [median_pct_change[i] for i in sorted_idx]
    median_dollar_change = [median_dollar_change[i] for i in sorted_idx]
    total_tax_change = [total_tax_change[i] for i in sorted_idx]

    # Custom color: anything above 0 is dark red, below 0 is green
    bar_colors = []
    for val in median_pct_change:
        if val > 0:
            bar_colors.append("#8B0000")  # dark red
        else:
            bar_colors.append("#228B22")  # professional green

    # Bar settings
    bar_height = 0.75
    fig_height = len(categories) * 0.8 + 1.2
    right_col_pad = 120  # more padding for right column
    fig, ax = plt.subplots(figsize=(17, fig_height))  # wider for right column

    y = np.arange(len(categories))

    # Draw bars
    ax.barh(
        y, median_pct_change, color=bar_colors, edgecolor='none',
        height=bar_height, alpha=0.92, linewidth=0, zorder=2
    )

    # Remove all spines and ticks for a clean look
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # Adjusted vertical spacing
    cat_offset = 0.18   # less space between category and median
    med_offset = -0.03  # median just below category
    count_offset = -0.23  # more space below median for parcels

    # For right column: position for total tax change
    max_abs = max(abs(min(median_pct_change)), abs(max(median_pct_change)))
    right_col_x = max_abs + right_col_pad

    # Add Net Change header at the top of the right column
    ax.text(
        right_col_x, len(categories) - 0.5, "Net Change", va='bottom', ha='left',
        fontsize=15, fontweight='bold', color='black', fontname='Arial'
    )

    for i, (cat, val, count, med_dol, tot_change) in enumerate(zip(categories, median_pct_change, counts, median_dollar_change, total_tax_change)):
        # Format median dollar and percent change together
        if med_dol >= 0:
            med_dol_str = f"${med_dol:,.0f}"
        else:
            med_dol_str = f"-${abs(med_dol):,.0f}"
        pct_str = f"{val:+.1f}%"
        median_combo = f"Median: {med_dol_str}, {pct_str}"

        # Position: right of bar for positive, left for negative
        if val < 0:
            xpos = val - 2.5
            ha = 'right'
        else:
            xpos = val + 2.5
            ha = 'left'
        # Category name (bold, bigger)
        ax.text(
            xpos, y[i]+cat_offset, cat, va='center', ha=ha,
            fontsize=14, fontweight='bold', color='#222',
            fontname='Arial'
        )
        # Median (dollar + percent, bold, black, just below category)
        ax.text(
            xpos, y[i]+med_offset, median_combo, va='center', ha=ha,
            fontsize=12, fontweight='bold', color='black',
            fontname='Arial'
        )
        # Count (bold, smaller, below median)
        ax.text(
            xpos, y[i]+count_offset, f"{count:,} parcels", va='center', ha=ha,
            fontsize=11, fontweight='bold', color='#888',
            fontname='Arial'
        )
        # Net change column, always right-aligned in a new column, black text, no "Total:"
        if tot_change >= 0:
            tot_change_str = f"${tot_change:,.0f}"
        else:
            tot_change_str = f"-${abs(tot_change):,.0f}"
        ax.text(
            right_col_x, y[i], tot_change_str, va='center', ha='left',
            fontsize=13, fontweight='bold', color='black',
            fontname='Arial'
        )

    # Set x limits for symmetry, make bars longer, and leave space for right column
    ax.set_xlim(-right_col_x, right_col_x + 60)

    # Remove axis labels/ticks
    ax.set_yticks([])
    ax.set_xticks([])

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Use output_summary to generate categories and percent increase/decrease, filtering to count > 50
# Filter to property_count > 50
summary_filtered = category_summary[category_summary['property_count'] > 50].copy()
# Sort by pct_increase_gt_threshold ascending (smallest percent increase first)
summary_sorted = summary_filtered.sort_values('pct_increase_gt_threshold', ascending=True)
categories_sorted = summary_sorted['PROPERTY_CATEGORY'].tolist()
pct_increase_sorted = summary_sorted['pct_increase_gt_threshold'].tolist()
pct_decrease_sorted = summary_sorted['pct_decrease_gt_threshold'].tolist()
# Convert to integers for display
pct_increase_int_sorted = [int(round(x)) for x in pct_increase_sorted]
pct_decrease_int_sorted = [int(round(x)) for x in pct_decrease_sorted]
y = np.arange(len(categories_sorted))
fig, ax = plt.subplots(figsize=(8, 6))
# Use specified colors
color_increase = "#8B0000"  # dark red
color_decrease = "#228B22"  # professional green
# Plot left (decrease) bars (green, to the left)
ax.barh(
    y, 
    [-v for v in pct_decrease_sorted], 
    color=color_decrease, 
    edgecolor='none', 
    height=0.7
)
# Plot right (increase) bars (red, to the right)
ax.barh(
    y, 
    pct_increase_sorted, 
    color=color_increase, 
    edgecolor='none', 
    height=0.7
)
# Add percent labels (integer, no decimals), smaller Arial font
for i, (inc, dec) in enumerate(zip(pct_increase_int_sorted, pct_decrease_int_sorted)):
    # Left side (decrease)
    if dec > 0:
        ax.text(
            -dec - 2, y[i], f"{dec}%", 
            va='center', ha='right', 
            fontsize=8, fontweight='normal', color=color_decrease, fontname='Arial'
        )
    # Right side (increase)
    if inc > 0:
        ax.text(
            inc + 2, y[i], f"{inc}%", 
            va='center', ha='left', 
            fontsize=8, fontweight='normal', color=color_increase, fontname='Arial'
        )
# Add category name at end of right bar, bold, smaller Arial, further from percent
for i, (cat, inc) in enumerate(zip(categories_sorted, pct_increase_sorted)):
    xpos = inc + 18 if inc > 0 else 18
    ax.text(
        xpos, y[i], cat, 
        va='center', ha='left', 
        fontsize=9, fontweight='bold', color='#222', fontname='Arial'
    )
# Remove all spines, ticks, and axis lines for minimalist look
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
# Remove grid, axis, and titles
ax.set_yticks([])
ax.set_xticks([])
ax.set_ylabel('')
ax.set_xlabel('')
ax.set_title('')
# Set xlim for symmetry
max_val = max(max(pct_increase_sorted), max(pct_decrease_sorted))
ax.set_xlim(-max_val-20, max_val+48)
# --- Add custom titles above left and right bars ---
# Make the titles a little bit bigger and closer to the center
title_fontsize = 10  # increased from 8
title_color = 'black'
title_fontweight = 'normal'
title_fontname = 'Arial'
# Compute center x for both titles, but offset slightly left/right of center
title_y = len(categories_sorted) - 0.2
# Left title (above left bars), closer to center
left_title_x = -max_val * 0.45
ax.text(
    left_title_x, title_y, 
    "Percent of parcels\ndecreasing >10%", 
    ha='center', va='bottom', fontsize=title_fontsize, fontweight=title_fontweight, 
    color=title_color, fontname=title_fontname, 
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15')
)
# Right title (above right bars), closer to center
right_title_x = max_val * 0.45
ax.text(
    right_title_x, title_y, 
    "Percent of parcels\nincreasing >10%", 
    ha='center', va='bottom', fontsize=title_fontsize, fontweight=title_fontweight, 
    color=title_color, fontname=title_fontname, 
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15')
)
plt.tight_layout()
plt.show()

In [ ]:
# display(
#     gdf[
#         gdf["PROPERTY_CATEGORY"].eq("duplex") &
#         gdf["tax_change_pct"].gt(40)
#     ][["taxable_value_land_nonschool","taxable_value_impr_nonschool","current_tax","new_tax","tax_change_pct"]]
# )

# #gdf.groupby("PROPERTY_CATEGORY")["tax_change_pct"].agg(["count","median"]).reset_index()

# census

In [ ]:
# Get census data for Denver County, CO (Denver) - FIPS code: 08031
print("📊 Loading Census data for Denver County, CO...")
df  = gdf
try:
    census_data, census_boundaries = get_census_data_with_boundaries(
        fips_code="08031",  # Denver County, CO
        year=2022
    )
    print("GOT CENSUS DATA")
    # Set CRS for census boundaries before merging
    census_boundaries = census_boundaries.set_crs(epsg=4326)  # Assuming WGS84 coordinate system
    
    # Ensure our parcel data is in the same CRS
    if df.crs != census_boundaries.crs:
        df = df.to_crs(census_boundaries.crs)
    
    # Merge census data with our parcel boundaries
    df = match_to_census_blockgroups(
        gdf=df,
        census_gdf=census_boundaries
    )
    
    print(f"✅ Census data integration complete!")
    print(f"Number of census block groups: {len(census_boundaries)}")
    print(f"Number of census data records: {len(census_data)}")
    print(f"Number of parcels with census data: {len(df)}")
    
    # Display new columns added
    census_cols = [col for col in df.columns if col in ['median_income', 'minority_pct', 'black_pct', 'total_pop', 'census_block_group']]
    print(f"Census columns added: {census_cols}")
    
except Exception as e:
    print(f"❌ Error loading census data: {e}")


In [ ]:
# Analyze tax impacts by income quintiles (similar to Spokane analysis)
print("📊 Analyzing tax impacts by neighborhood income quintiles...")

if 'median_income' in df.columns:
    # Filter out parcels with missing or non-positive income data
    df_with_income = df[(df['median_income'].notna()) & (df['median_income'] > 0)].copy()
    
    # Create income quintiles
    df_with_income['income_quintile'] = pd.qcut(
        df_with_income['median_income'], 
        5, 
        labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"]
    )
    
    # Calculate summary statistics by quintile
    quintile_summary = df_with_income.groupby('income_quintile').agg({
        'tax_change': ['count', 'mean', 'median'],
        'tax_change_pct': ['mean', 'median'],
        'median_income': 'mean',
        'current_tax': 'mean'
    }).round(2)
    
    # Flatten column names
    quintile_summary.columns = ['_'.join(col).strip() for col in quintile_summary.columns]
    quintile_summary = quintile_summary.reset_index()
    
    print(f"✅ Income quintile analysis complete")
    print(f"Parcels with income data: {len(df_with_income):,} ({len(df_with_income)/len(df)*100:.1f}%)")
    
    display(quintile_summary)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Median tax change percentage by income quintile
    quintile_data = quintile_summary.copy()
    quintile_data['median_tax_change_pct'] = quintile_data['tax_change_pct_median']
    
    bars1 = ax1.bar(
        quintile_data['income_quintile'],
        quintile_data['median_tax_change_pct'],
        color='steelblue',
        alpha=0.7
    )
    
    ax1.set_title('Median Tax Change % by Income Quintile', fontweight='bold', pad=20)
    ax1.set_ylabel('Median Tax Change (%)')
    ax1.set_xlabel('Income Quintile')
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars1, quintile_data['median_tax_change_pct']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Average neighborhood income by quintile
    bars2 = ax2.bar(
        quintile_data['income_quintile'],
        quintile_data['median_income_mean'],
        color='green',
        alpha=0.7
    )
    
    ax2.set_title('Average Neighborhood Income by Quintile', fontweight='bold', pad=20)
    ax2.set_ylabel('Average Median Income ($)')
    ax2.set_xlabel('Income Quintile')
    ax2.grid(True, alpha=0.3)
    
    # Format y-axis as currency
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Add value labels on bars
    for bar, val in zip(bars2, quintile_data['median_income_mean']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                f'${val:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Census income data not available - skipping quintile analysis")


In [ ]:
# Scatter plot analysis: Income vs Tax Impact (similar to Spokane)
print("📈 Creating scatter plot analysis...")

if 'median_income' in df.columns and 'tax_change_pct' in df.columns:
    # Filter data for meaningful analysis
    plot_data = df[
        (df['median_income'].notna()) & 
        (df['median_income'] > 0) & 
        (df['tax_change_pct'].notna()) &
        (df['tax_change_pct'].abs() < 100)  # Remove extreme outliers
    ].copy()
    
    # Create block group level summary for cleaner visualization
    if 'census_block_group' in df.columns:
        block_summary = plot_data.groupby('census_block_group').agg({
            'median_income': 'first',  # Same for all parcels in block group
            'tax_change_pct': 'median',
            'tax_change': 'median',
            'current_tax': 'sum',
            'total_pop': 'first'
        }).reset_index()
        
        plot_data_agg = block_summary
        size_col = 'total_pop'
        title_suffix = "by Census Block Group"
    else:
        # Use individual parcels if no block group data
        plot_data_agg = plot_data.sample(min(1000, len(plot_data)))  # Sample for performance
        size_col = 'current_tax'
        title_suffix = "by Individual Parcels (Sample)"
    
    # Create the scatter plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create scatter plot with size based on population or tax
    scatter = ax.scatter(
        plot_data_agg['median_income'],
        plot_data_agg['tax_change_pct'],
        s=plot_data_agg[size_col] / plot_data_agg[size_col].max() * 200 + 20,
        alpha=0.6,
        c=plot_data_agg['tax_change_pct'],
        cmap='RdYlBu_r',
        edgecolors='black',
        linewidth=0.5
    )
    
    # Add trend line
    from scipy import stats
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        plot_data_agg['median_income'], 
        plot_data_agg['tax_change_pct']
    )
    line_x = np.array([plot_data_agg['median_income'].min(), plot_data_agg['median_income'].max()])
    line_y = slope * line_x + intercept
    ax.plot(line_x, line_y, 'r--', alpha=0.8, linewidth=2, 
            label=f'Trend (R² = {r_value**2:.3f})')
    
    # Formatting
    ax.set_xlabel('Neighborhood Median Income ($)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Median Tax Change (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Tax Impact vs. Neighborhood Income {title_suffix}', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Format x-axis as currency
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Tax Change (%)', fontweight='bold')
    
    # Add legend
    ax.legend()
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print correlation statistics
    correlation = plot_data_agg['median_income'].corr(plot_data_agg['tax_change_pct'])
    print(f"\\n📊 Correlation Analysis:")
    print(f"- Correlation coefficient: {correlation:.3f}")
    print(f"- R-squared: {r_value**2:.3f}")
    print(f"- P-value: {p_value:.3e}")
    print(f"- Sample size: {len(plot_data_agg):,} observations")
    
    if correlation > 0:
        print("📈 Positive correlation: Higher income areas tend to have higher tax increases")
    elif correlation < 0:
        print("📉 Negative correlation: Higher income areas tend to have lower tax increases")
    else:
        print("➡️  No clear correlation between income and tax changes")
        
else:
    print("❌ Required data not available for scatter plot analysis")


In [ ]:
# Final Summary Analysis
print("📋 DENVER LVT POLICY ANALYSIS SUMMARY")
print("=" * 60)

# Overall tax impact summary
total_parcels = len(df)
total_current_tax = df['current_tax'].sum()
total_tax_change = df['tax_change'].sum()
total_new_tax = total_current_tax + total_tax_change
category_col = "model_group"

print(f"\n🏠 OVERALL IMPACT:")
print(f"- Total parcels analyzed: {total_parcels:,}")
print(f"- Current total tax revenue: ${total_current_tax:,.2f}")
print(f"- Total tax change: ${total_tax_change:,.2f}")
print(f"- New total tax revenue: ${total_new_tax:,.2f}")
print(f"- Net change: {(total_tax_change/total_current_tax)*100:+.2f}%")

# Winners and losers
tax_increases = (df['tax_change'] > 0).sum()
tax_decreases = (df['tax_change'] < 0).sum()
no_change = (df['tax_change'] == 0).sum()

print(f"\n📊 DISTRIBUTION OF IMPACTS:")
print(f"- Properties with tax increases: {tax_increases:,} ({tax_increases/total_parcels*100:.1f}%)")
print(f"- Properties with tax decreases: {tax_decreases:,} ({tax_decreases/total_parcels*100:.1f}%)")
print(f"- Properties with no change: {no_change:,} ({no_change/total_parcels*100:.1f}%)")

# Median impacts
median_tax_change = df['tax_change'].median()
median_tax_change_pct = df['tax_change_pct'].median()

print(f"\n📈 TYPICAL IMPACTS:")
print(f"- Median tax change: ${median_tax_change:.2f}")
print(f"- Median tax change percentage: {median_tax_change_pct:.2f}%")

# Census-based equity analysis (if available)
if 'median_income' in df.columns:
    print(f"\n🏘️  EQUITY ANALYSIS:")
    
    # Low vs high income areas
    df_with_income = df[(df['median_income'].notna()) & (df['median_income'] > 0)]
    low_income_threshold = df_with_income['median_income'].quantile(0.33)
    high_income_threshold = df_with_income['median_income'].quantile(0.67)
    
    low_income_parcels = df_with_income[df_with_income['median_income'] <= low_income_threshold]
    high_income_parcels = df_with_income[df_with_income['median_income'] >= high_income_threshold]
    
    low_income_median_change = low_income_parcels['tax_change_pct'].median()
    high_income_median_change = high_income_parcels['tax_change_pct'].median()
    
    print(f"- Low-income areas (bottom 33%): {low_income_median_change:.2f}% median tax change")
    print(f"- High-income areas (top 33%): {high_income_median_change:.2f}% median tax change")
    
    if low_income_median_change < high_income_median_change:
        print("  → Policy appears progressive (lower burden on low-income areas)")
    elif low_income_median_change > high_income_median_change:
        print("  → Policy appears regressive (higher burden on low-income areas)")
    else:
        print("  → Policy appears neutral across income levels")

# Property type analysis summary
if category_col in df.columns:
    print(f"\n🏢 PROPERTY TYPE IMPACTS:")
    
    # Find the categories with biggest increases and decreases
    cat_summary = df.groupby(category_col)['tax_change'].agg(['sum', 'median', 'count']).round(2)
    cat_summary = cat_summary.sort_values('sum', ascending=False)
    
    print("Top 3 categories by total tax increase:")
    for i, (cat, row) in enumerate(cat_summary.head(3).iterrows()):
        print(f"  {i+1}. {cat}: ${row['sum']:,.0f} total, ${row['median']:.0f} median ({row['count']:,} properties)")
    
    print("\nTop 3 categories by total tax decrease:")
    for i, (cat, row) in enumerate(cat_summary.tail(3).iterrows()):
        print(f"  {i+1}. {cat}: ${row['sum']:,.0f} total, ${row['median']:.0f} median ({row['count']:,} properties)")

print(f"\n" + "=" * 60)
print("📝 NOTES:")
print("- This analysis uses sample/placeholder tax change data for demonstration")
print("- Replace the tax change calculations with your actual LVT policy model")
print("- Census data integration provides demographic context for equity analysis")
print("- Visualizations above show the distributional impacts across income and property types")

print(f"\n✅ Analysis complete! Review the visualizations above for detailed insights.")


In [ ]:
# Calculate and report the sum of the absolute difference between current_tax and new_tax,
# and what percent of the sum of current_tax that represents.
# Calculate absolute difference per parcel

df_b = df.copy()

df_b['abs_tax_diff'] = (df_b['current_tax'] - df_b['new_tax']).abs()
# Sum absolute differences
total_abs_tax_diff = df_b['abs_tax_diff'].sum()
# Calculate what percent of total current tax that represents
percent_of_current = (total_abs_tax_diff / total_current_tax) * 100 if total_current_tax != 0 else np.nan
print(f"Sum of absolute value of current_tax minus new_tax: ${total_abs_tax_diff:,.2f}")
print(f"That is {percent_of_current:.2f}% of the sum of current_tax.")

In [ ]:
gdf[gdf["assr_impr_value"].le(0) & gdf["is_exempt"].eq(False)]["assr_land_value"].sum()

In [ ]:
# Undeveloped and underdeveloped land summary
result = analyze_land_by_improvement_share(
    gdf,
    land_value_col='assr_land_value',
    improvement_value_col='assr_impr_value',
)

total = result['total_adjusted_land_value']
cats = {c['category']: c for c in result['categories']}

labels = {
    '0% improvement':               'Undeveloped (0% improvement)',
    '<10% improvement (excl. 0%)':  'Underdeveloped: < 10% improvement',
    '10-25% improvement':           'Underdeveloped: 10-25% improvement',
    '>0%-25% improvement':          'Underdeveloped (all)'
}

print(f"Total non-exempt assessed land value: ${total:>20,.0f}")
print()
print(f"{'Category':<40} {'Parcels':>8}  {'Land Value':>18}  {'% of Total':>10}")
print("-" * 82)
for key, label in labels.items():
    c = cats[key]
    print(f"{label:<40} {c['parcel_count']:>8,}  ${c['adjusted_land_value']:>17,.0f}  {c['share_of_total_land_value_pct']:>9.2f}%")

c_both = {}
for key in ['0% improvement', '>0%-25% improvement']:
    c = cats[key]
    for k in c:
        if k not in c_both:
            c_both[k] = c[k]
        else:
            c_both[k] += c[k]

c = c_both
label = "Undeveloped AND Underdeveloped"
print(f"{label:<40} {c['parcel_count']:>8,}  ${c['adjusted_land_value']:>17,.0f}  {c['share_of_total_land_value_pct']:>9.2f}%")

In [ ]:
# Tag parcels with development status and write to geoparquet
impr_share = gdf['assr_impr_value'] / (gdf['assr_land_value'] + gdf['assr_impr_value'])
impr_share = impr_share.fillna(0)

def _dev_tag(row_share, row_impr):
    if row_impr == 0:
        return 'undeveloped'
    elif row_share < 0.10:
        return 'underdeveloped_lt10'
    elif row_share < 0.25:
        return 'underdeveloped_10_25'
    elif row_share < 0.50:
        return 'underdeveloped_25_50'
    return 'developed'

gdf['development_status'] = [
    _dev_tag(s, i) for s, i in zip(impr_share, gdf['assr_impr_value'])
]

out_path = 'data/denver/parcels_with_development_status.parquet'
gdf.to_parquet(out_path, index=False)
print(f"Written {len(gdf):,} parcels to {out_path}")
print(gdf['development_status'].value_counts().to_string())


In [ ]:
# TODO: Export standardized CSV — denver
# This cell needs to be completed manually before running.
#
# Denver has a dual-levy structure (school + non-school). The current_tax and new_tax columns are combined totals (school + nonschool). The land_millage and imp_millage expressions average school and nonschool rates — verify these variable names (land_mills_s, impr_mills_s, land_mills_ns, impr_mills_ns) exist in scope when this cell runs. Also verify that taxable_land_value column exists in gdf_lvt after model runs.
#
# Template:
# import sys
# sys.path.insert(0, '..')
# from lvt_utils import save_standard_export
#
# save_standard_export(
#     df=<final_df_variable>,
#     city='denver',
#     output_path='../analysis/data/denver.csv',
#     model_type='split_rate:4.0',  # update as needed
#     land_millage=land_millage,
#     improvement_millage=improvement_millage,
# )
